# Web Scraping IMDB movie rating

2. IMDb is a JavaScript website 🚨

IMDb loads movies like this:

Initial HTML → JavaScript runs → Movies appear

So:

❌ With requests + headers:
You get → initial HTML only
(no movies)

In [55]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

url = "https://www.imdb.com/chart/top"

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)  # wait for JS to load

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

movies = soup.select("li.ipc-metadata-list-summary-item")

🧠 Why Selenium and NOT headers?
🔹 1. Headers = only identity (NOT behavior)

When you do:

requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

👉 You are just telling the server:

“Hey, I am Chrome”

But in reality:

❌ No JavaScript runs

❌ No dynamic content loads

❌ No scrolling / rendering

✅ With Selenium:
Browser opens → JS runs → page fully loads → movies visible

👉 Result:

len(movies) = 250

In [56]:
print(len(movies))

250


In [ ]:
movie_data = []

for movie in movies:
    # Title
    title = movie.select_one("h3").text.strip()

    # Year (
    metadata_items = movie.select("span.cli-title-metadata-item")
    year = metadata_items[0].text.strip() if len(metadata_items) > 0 else "N/A"

    # Rating
    rating_tag = movie.select_one(".ipc-rating-star--rating")
    rating = rating_tag.text.strip() if rating_tag else "N/A"

    movie_data.append({
        "Title": title,
        "Year": year,
        "Rating": rating
    })

print(movie_data)

[{'Title': 'The Shawshank Redemption', 'Year': '1994', 'Rating': '9.3'}, {'Title': 'The Godfather', 'Year': '1972', 'Rating': '9.2'}, {'Title': 'The Dark Knight', 'Year': '2008', 'Rating': '9.1'}, {'Title': 'The Godfather Part II', 'Year': '1974', 'Rating': '9.0'}, {'Title': '12 Angry Men', 'Year': '1957', 'Rating': '9.0'}, {'Title': 'The Lord of the Rings: The Return of the King', 'Year': '2003', 'Rating': '9.0'}, {'Title': "Schindler's List", 'Year': '1993', 'Rating': '9.0'}, {'Title': 'The Lord of the Rings: The Fellowship of the Ring', 'Year': '2001', 'Rating': '8.9'}, {'Title': 'Pulp Fiction', 'Year': '1994', 'Rating': '8.8'}, {'Title': 'The Good, the Bad and the Ugly', 'Year': '1966', 'Rating': '8.8'}, {'Title': 'The Lord of the Rings: The Two Towers', 'Year': '2002', 'Rating': '8.8'}, {'Title': 'Forrest Gump', 'Year': '1994', 'Rating': '8.8'}, {'Title': 'Fight Club', 'Year': '1999', 'Rating': '8.8'}, {'Title': 'Inception', 'Year': '2010', 'Rating': '8.8'}, {'Title': 'Star Wars: 

In [61]:
import csv

with open("movies.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["Title", "Year", "Rating"])
    
    writer.writeheader()  # column names
    writer.writerows(movie_data)  # data

print("Data saved to movies.csv ✅")

Data saved to movies.csv ✅


🚀 When to use what?
✅ Use headers (requests) when:

Data is already in HTML

Static websites

✅ Use Selenium when:

Data loads after page load

Buttons / scrolling needed

Site uses JS (like IMDb)